# Inférence — LSTM
Charge un modèle entraîné, génère les prédictions sur train/val/test et les sauvegarde en parquet.

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

from LSTM import LSTM, make_loaders, set_seed
from settings import (
    DATA_DIR, MODEL_DIR, OUTPUT_DIR,
    PAIRS, N_TARGETS, TARGET_COLS,
    CONTEXT_LENGTH, BATCH_SIZE,
    SEED, DEBUG,
)

NUM_LAYERS     = 1
MODEL_FILENAME = f'lstm_{NUM_LAYERS}l.pt'

if DEBUG:
    print('[DEBUG] Mode debug activé — 1% des données')

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}  |  Modèle : {MODEL_FILENAME}')

## 1. Chargement des données

In [ ]:
splits = {}
for name in ('train', 'validation', 'test'):
    path = DATA_DIR / f'{name}_features_crypto.parquet'
    try:
        splits[name] = pd.read_parquet(path).ffill()
        print(f'{name:12s} {splits[name].shape}  {path.name}')
    except FileNotFoundError:
        print(f'[MANQUANT] {path}')

if len(splits) < 3:
    raise FileNotFoundError(f'Données incomplètes — placez les parquets dans {DATA_DIR}')

if DEBUG:
    splits = {k: v.iloc[:max(int(len(v) * 0.01), CONTEXT_LENGTH + 1)] for k, v in splits.items()}
    print(f'\n[DEBUG] Données réduites à 1% : { {k: v.shape for k, v in splits.items()} }')

In [ ]:
train_df, val_df, test_df = splits['train'], splits['validation'], splits['test']
feature_cols = [c for c in train_df.columns if c not in TARGET_COLS]

print(f'Features : {len(feature_cols)}  |  Cibles : {N_TARGETS}')
print(f'\nAperçu train — premières cibles :')
train_df[TARGET_COLS].describe().round(4)

In [ ]:
train_loader, val_loader, test_loader, n_features = make_loaders(
    train_df, val_df, test_df, feature_cols, TARGET_COLS,
    seq_len=CONTEXT_LENGTH, batch_size=BATCH_SIZE, seed=SEED,
)
print(f'n_features={n_features}  |  séquences train={len(train_loader.dataset):,}')

## 2. Chargement du modèle

In [ ]:
model_path  = MODEL_DIR / MODEL_FILENAME
params_path = MODEL_DIR / f'lstm_{NUM_LAYERS}l_params.json'

try:
    with open(params_path) as f:
        saved = json.load(f)
    hidden_size = saved['hidden_size']
    print(f'Hyperparamètres chargés : hidden_size={hidden_size}')
except FileNotFoundError:
    raise FileNotFoundError(
        f'Paramètres introuvables : {params_path} — lancez hyperparametres.ipynb en premier'
    )

try:
    model = LSTM(
        n_features=n_features, n_targets=N_TARGETS,
        hidden_size=hidden_size, num_layers=NUM_LAYERS,
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device).eval()
    print(f'Modèle chargé depuis {model_path}')
    print(model)
except FileNotFoundError:
    raise FileNotFoundError(
        f'Modèle introuvable : {model_path} — lancez hyperparametres.ipynb en premier'
    )

## 3. Inférence

In [ ]:
def run_inference(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for x, _ in tqdm(loader, leave=False):
            preds.append(model(x.to(device)).cpu().numpy())
    return np.concatenate(preds, axis=0)


pred_cols = [f'{p}_pred_bps' for p in PAIRS]
loaders   = {'train': (train_loader, train_df), 'validation': (val_loader, val_df), 'test': (test_loader, test_df)}
all_preds = {}

for split, (loader, ref_df) in loaders.items():
    preds_np = run_inference(model, loader, device)
    idx      = ref_df.index[CONTEXT_LENGTH - 1:]
    preds_df = pd.DataFrame(preds_np, index=idx, columns=pred_cols)
    all_preds[split] = preds_df

    out_path = OUTPUT_DIR / f'{split}_lstm_{NUM_LAYERS}l_predictions.parquet'
    preds_df.to_parquet(out_path, engine='pyarrow')
    print(f'{split:12s} {preds_df.shape}  -> {out_path.name}')

## 4. Visualisation — Prédictions vs Cibles

In [ ]:
MAX_POINTS = 5000

fig, axes = plt.subplots(N_TARGETS, 1, figsize=(18, 5 * N_TARGETS), sharex=True)
fig.suptitle(f'Prédictions vs Cibles — LSTM {NUM_LAYERS} couche(s)', fontsize=14, y=1.001)

COLORS = {'train': '#2196F3', 'validation': '#4CAF50', 'test': '#FF5722'}

for ax, (pair, target_col, pred_col) in zip(axes, zip(PAIRS, TARGET_COLS, pred_cols)):
    for split, preds_df in all_preds.items():
        ref_df  = loaders[split][1]
        tgt_df  = ref_df[target_col].iloc[CONTEXT_LENGTH - 1:]
        step    = max(1, len(preds_df) // MAX_POINTS)
        color   = COLORS[split]

        ax.plot(preds_df.index[::step], preds_df[pred_col][::step],
                color=color, lw=0.9, alpha=0.85, label=f'Prédiction ({split})')
        ax.plot(tgt_df.index[::step], tgt_df.values[::step],
                color=color, lw=0.9, alpha=0.4, ls='--', label=f'Réel ({split})')

    ax.set_title(pair, fontsize=11)
    ax.set_ylabel('Volatilité (BPS)')
    ax.grid(True, alpha=0.3)
    if ax == axes[0]:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=8)

axes[-1].set_xlabel('Date')
plt.tight_layout()

fig_path = OUTPUT_DIR / f'predictions_{NUM_LAYERS}l.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Graphique sauvegardé : {fig_path}')
plt.show()

## 5. Métriques rapides

In [ ]:
rows = []
for split, preds_df in all_preds.items():
    ref_df = loaders[split][1]
    tgt    = ref_df[TARGET_COLS].iloc[CONTEXT_LENGTH - 1:].values
    pred   = preds_df.values
    rmse_per_pair = np.sqrt(np.mean((pred - tgt) ** 2, axis=0))
    for pair, rmse in zip(PAIRS, rmse_per_pair):
        rows.append({'split': split, 'pair': pair, 'RMSE (BPS)': round(float(rmse), 4)})

metrics = pd.DataFrame(rows).pivot(index='pair', columns='split', values='RMSE (BPS)')
metrics.loc['MOYENNE'] = metrics.mean()
metrics